<h2 align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
0. Import Libraries

</font>
</h2>

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from music21 import converter, instrument, note, chord, stream
from keras.models import Sequential
from keras.layers import LSTM, Dropout, Dense, Activation
from keras.callbacks import ModelCheckpoint, EarlyStopping, CSVLogger
import random
import os

# تنظیم فونت فارسی برای matplotlib
plt.rcParams['font.family'] = 'Vazir'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.unicode_minus'] = False

<h2 align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
1. Data Loading
</font>
</h2>

In [8]:
def get_notes():
    notes = []
    midi = converter.parse('original_metheny.mid')  
    notes_to_parse = None
    try:
        s2 = instrument.partitionByInstrument(midi)
        notes_to_parse = s2.parts[0].recurse()
    except:
        notes_to_parse = midi.flat.notes
    
    for element in notes_to_parse:
        if isinstance(element, note.Note):
            notes.append(str(element.pitch))
        elif isinstance(element, chord.Chord):
            notes.append('.'.join(str(n) for n in element.normalOrder))
    return notes

<h2 align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
2. Preprocessing
</font>
</h2>

In [9]:
notes = get_notes()
print(f"تعداد کل نت‌ها و آکوردها: {len(notes)}")

pitchnames = sorted(set(notes))
n_vocab = len(pitchnames)
print(f"اندازه واژگان منحصربه‌فرد: {n_vocab}")

تعداد کل نت‌ها و آکوردها: 647
اندازه واژگان منحصربه‌فرد: 120


In [10]:
note_to_int = {note: number for number, note in enumerate(pitchnames)}

sequence_length = 100
network_input = []
network_output = []

for i in range(len(notes) - sequence_length):
    seq_in = notes[i:i + sequence_length]
    seq_out = notes[i + sequence_length]
    network_input.append([note_to_int[ch] for ch in seq_in])
    network_output.append(note_to_int[seq_out])

n_patterns = len(network_input)
network_input = np.reshape(network_input, (n_patterns, sequence_length, 1))
network_input = network_input / float(n_vocab)
network_output = np.eye(n_vocab)[network_output]

print(f"تعداد الگوهای آموزشی: {n_patterns}")

تعداد الگوهای آموزشی: 547


<h2  align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
3. Model Development
</font>
</h2>

In [11]:
model = Sequential()
model.add(LSTM(512, input_shape=(network_input.shape[1], network_input.shape[2]), return_sequences=True))
model.add(Dropout(0.3))
model.add(LSTM(512, return_sequences=True))
model.add(Dropout(0.3))
model.add(LSTM(512))
model.add(Dense(256))
model.add(Dropout(0.3))
model.add(Dense(n_vocab))
model.add(Activation('softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam')
model.summary()

d:\conda_envs\quera\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 100, 512)       │     1,052,672 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 100, 512)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 100, 512)       │     2,099,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 100, 512)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 512)            │     2,099,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 120)            │        30,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 120)            │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,413,240 (20.65 MB)

 Trainable params: 5,413,240 (20.65 MB)

 Non-trainable params: 0 (0.00 B)

<h2  align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
4. Training
</font>
</h2>

In [12]:
checkpoint = ModelCheckpoint('best_model.h5', monitor='loss', save_best_only=True, verbose=1)
early_stop = EarlyStopping(monitor='loss', patience=10, verbose=1)
csv_logger = CSVLogger('training_log.csv', append=True)

history = model.fit(network_input, network_output, epochs=50, batch_size=64, 
                    callbacks=[checkpoint, early_stop, csv_logger], verbose=1)

Epoch 1/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - loss: 4.7572
Epoch 1: loss improved from inf to 4.71749, saving model to best_model.h5


9/9 ━━━━━━━━━━━━━━━━━━━━ 22s 2s/step - loss: 4.7533
Epoch 2/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - loss: 4.6285
Epoch 2: loss improved from 4.71749 to 4.62249, saving model to best_model.h5


9/9 ━━━━━━━━━━━━━━━━━━━━ 18s 2s/step - loss: 4.6279
Epoch 3/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - loss: 4.4938
Epoch 3: loss improved from 4.62249 to 4.52392, saving model to best_model.h5


9/9 ━━━━━━━━━━━━━━━━━━━━ 18s 2s/step - loss: 4.4968
Epoch 4/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - loss: 4.4865
Epoch 4: loss improved from 4.52392 to 4.50084, saving model to best_model.h5


9/9 ━━━━━━━━━━━━━━━━━━━━ 20s 2s/step - loss: 4.4879
Epoch 5/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - loss: 4.4478
Epoch 5: loss improved from 4.50084 to 4.47370, saving model to best_model.h5


...

9/9 ━━━━━━━━━━━━━━━━━━━━ 32s 3s/step - loss: 0.7578
Epoch 44/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - loss: 0.7269
Epoch 44: loss improved from 0.76586 to 0.76300, saving model to best_model.h5


9/9 ━━━━━━━━━━━━━━━━━━━━ 31s 3s/step - loss: 0.7305
Epoch 45/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - loss: 0.6705
Epoch 45: loss improved from 0.76300 to 0.68490, saving model to best_model.h5


9/9 ━━━━━━━━━━━━━━━━━━━━ 31s 3s/step - loss: 0.6720
Epoch 46/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - loss: 0.6698
Epoch 46: loss did not improve from 0.68490
9/9 ━━━━━━━━━━━━━━━━━━━━ 31s 3s/step - loss: 0.6721
Epoch 47/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - loss: 0.6199
Epoch 47: loss improved from 0.68490 to 0.64056, saving model to best_model.h5


9/9 ━━━━━━━━━━━━━━━━━━━━ 32s 4s/step - loss: 0.6219
Epoch 48/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - loss: 0.6023
Epoch 48: loss did not improve from 0.64056
9/9 ━━━━━━━━━━━━━━━━━━━━ 31s 3s/step - loss: 0.6063
Epoch 49/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - loss: 0.6196
Epoch 49: loss did not improve from 0.64056
9/9 ━━━━━━━━━━━━━━━━━━━━ 25s 3s/step - loss: 0.6219
Epoch 50/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - loss: 0.5568
Epoch 50: loss improved from 0.64056 to 0.62431, saving model to best_model.h5


9/9 ━━━━━━━━━━━━━━━━━━━━ 21s 2s/step - loss: 0.5635


<h2  align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
5. Evaluation
</font>
</h2>



In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.history['loss'], label='خطای آموزش')
plt.xlabel('دوره (Epoch)')
plt.ylabel('خطا (Loss)')
plt.title('نمودار  آموزش مدل')
plt.legend()
plt.savefig('training_loss.png')
plt.show()

</font>
<h2  align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
6. Music Generation
</font>
</h2>



In [14]:
start = np.random.randint(0, len(network_input)-1)
pattern = network_input[start]

prediction_output = []
for note_index in range(200):
    prediction_input = np.reshape(pattern, (1, len(pattern), 1))
    prediction_input = prediction_input / float(n_vocab)
    prediction = model.predict(prediction_input, verbose=0)
    index = np.argmax(prediction)
    result = pitchnames[index]
    prediction_output.append(result)
    pattern = np.append(pattern, index)
    pattern = pattern[1:]

print("دنباله نت‌های تولیدشده:")
print(prediction_output[:50], "...")

دنباله نت‌های تولیدشده:
['7.9.11.1.3', '7.9.11.1.3', '7.9.11.1.3', 'A1', 'A1', 'A1', '4.7.9.11.0', 'D4', '11.0.4.7', '11.0.4.7', '4.6.9.0', '2.6.9', '2.6.9', '11.2.4.7', '4.7.9.11.0', '4.7.9.11.0', '4.7.9.11.0', '0.2.4.7', '4.5.7.11', '4.5.7.11', 'A2', '1.4.6.8.9', '1.4.6.8.9', '3.6.9.11.0', '11.2.4.6.7', '11.2.4.6.7', '7.9.10.1.3', '9.0.2.4.5', '4.5.7.9.11', '11.0.2.4.7', '7.9.11.2', '7.9.2', '7.8.10', 'D4', 'E-4', 'D4', '11.0.4.7', '11.0.4.7', '4.6.9.0', '2.6.9', 'C4', '2.6.9', '11.2.4.7', '4.7.9.11.0', '4.7.9.11.0', '0.2.4.7', '0.2.4.7', '4.5.7.11', '4.5.7.11', 'A2'] ...


In [15]:
offset = 0
output_notes = []
for pattern in prediction_output:
    if ('.' in pattern) or pattern.isdigit():
        notes_in_chord = pattern.split('.')
        notes_ = []
        for current_note in notes_in_chord:
            new_note = note.Note(int(current_note))
            new_note.storedInstrument = instrument.Piano()
            notes_.append(new_note)
        new_chord = chord.Chord(notes_)
        new_chord.offset = offset
        output_notes.append(new_chord)
    else:
        new_note = note.Note(pattern)
        new_note.offset = offset
        new_note.storedInstrument = instrument.Piano()
        output_notes.append(new_note)
    offset += 0.5

midi_stream = stream.Stream(output_notes)
output_file = 'test_output.mid'
midi_stream.write('midi', fp=output_file)

print(f"فایل MIDI با نام '{output_file}' با موفقیت تولید شد!")

فایل MIDI با نام 'test_output.mid' با موفقیت تولید شد!


</font>
<h2  align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
6. Play Music
</font>
</h2>



In [18]:
try:
    import pygame
    pygame.init()
    pygame.mixer.init()
    pygame.mixer.music.load(output_file)
    pygame.mixer.music.play()
    print("در حال پخش موسیقی تولیدشده...")
except Exception as e:
    print("امکان پخش خودکار MIDI وجود ندارد. لطفاً فایل را به صورت دستی در یک پخش‌کننده MIDI باز کنید.")
    print("خطا:", e)

d:\conda_envs\quera\lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.9.25)
Hello from the pygame community. https://www.pygame.org/contribute.html
در حال پخش موسیقی تولیدشده...
